In [5]:
import re
import numpy as np
import pandas as pd


# ============================================================
# ATTRIBUTE EXTRACTION
# ============================================================

KNOWN_BRANDS = [
    "apple",
    "samsung",
    "dell",
    "hp",
    "hewlett packard",
    "lenovo",
    "asus",
    "acer",
    "microsoft",
    "sony",
    "lg",
    "logitech",
    "intel",
    "amd",
    "nvidia",
    "evga",
    "synology",
    "qnap",
    "western digital",
    "seagate",
    "sandisk",
    "kingston",
    "corsair",
    "canon",
    "nikon",
    "epson",
    "brother",
    "panasonic",
    "bose",
    "jbl",
    "anker",
    "motorola",
    "garmin",
    "netgear",
    "tp-link",
    "hypertherm",
    "silverstone",
    "devialet",
    "victron",
]


def extract_number(text, pattern):
    match = re.search(
        pattern,
        str(text).lower(),
        flags=re.IGNORECASE,
    )

    if match:
        try:
            return float(match.group(1))
        except:
            return 0.0

    return 0.0


def extract_brand(text):

    text = str(text).lower()

    for brand in KNOWN_BRANDS:

        if re.search(
            rf"\b{re.escape(brand)}\b",
            text,
        ):
            return brand

    return "unknown"


def enrich_title_features(df):

    df = df.copy()

    title = (
        df["title"]
        .fillna("")
        .astype(str)
    )

    # --------------------------------------------------------
    # BRAND
    # --------------------------------------------------------

    df["extracted_brand"] = title.apply(
        extract_brand
    )

    # --------------------------------------------------------
    # RAM / STORAGE
    # --------------------------------------------------------

    df["gb_value"] = title.apply(
        lambda x: extract_number(
            x,
            r"(\d+(?:\.\d+)?)\s*gb\b",
        )
    )

    df["tb_value"] = title.apply(
        lambda x: extract_number(
            x,
            r"(\d+(?:\.\d+)?)\s*tb\b",
        )
    )

    # Convert TB to approximate GB
    df["storage_capacity_gb"] = (
        df["gb_value"]
        + df["tb_value"] * 1024
    )

    # --------------------------------------------------------
    # SCREEN SIZE
    # --------------------------------------------------------

    df["screen_size_inches"] = title.apply(
        lambda x: extract_number(
            x,
            r'(\d+(?:\.\d+)?)\s*(?:inch|inches|")',
        )
    )

    # --------------------------------------------------------
    # WATTAGE
    # --------------------------------------------------------

    df["wattage"] = title.apply(
        lambda x: extract_number(
            x,
            r"(\d+(?:\.\d+)?)\s*(?:w|watt|watts)\b",
        )
    )

    # --------------------------------------------------------
    # VOLTAGE
    # --------------------------------------------------------

    df["voltage"] = title.apply(
        lambda x: extract_number(
            x,
            r"(\d+(?:\.\d+)?)\s*(?:v|volt|volts)\b",
        )
    )

    # --------------------------------------------------------
    # VA
    # --------------------------------------------------------

    df["va_rating"] = title.apply(
        lambda x: extract_number(
            x,
            r"(\d+(?:\.\d+)?)\s*va\b",
        )
    )

    # --------------------------------------------------------
    # LITRES
    # --------------------------------------------------------

    df["capacity_liters"] = title.apply(
        lambda x: extract_number(
            x,
            r"(\d+(?:\.\d+)?)\s*(?:l|liter|litre|liters|litres)\b",
        )
    )

    # --------------------------------------------------------
    # PACK / QUANTITY
    # --------------------------------------------------------

    df["pack_count"] = title.apply(
        lambda x: extract_number(
            x,
            r"(\d+)\s*[- ]?(?:pack|pcs|piece|pieces|count|ct)\b",
        )
    )

    # --------------------------------------------------------
    # YEAR
    # --------------------------------------------------------

    def extract_year(x):

        match = re.search(
            r"\b(20(?:1[0-9]|2[0-6]))\b",
            str(x),
        )

        if match:
            return int(match.group(1))

        return 0

    df["model_year"] = title.apply(
        extract_year
    )

    # --------------------------------------------------------
    # BAY COUNT
    # --------------------------------------------------------

    df["bay_count"] = title.apply(
        lambda x: extract_number(
            x,
            r"(\d+)\s*[- ]?bay\b",
        )
    )

    # --------------------------------------------------------
    # CPU SIGNALS
    # --------------------------------------------------------

    lower_title = title.str.lower()

    df["has_i3"] = lower_title.str.contains(
        r"\bi3[-\s]?\d*",
        regex=True,
    ).astype(int)

    df["has_i5"] = lower_title.str.contains(
        r"\bi5[-\s]?\d*",
        regex=True,
    ).astype(int)

    df["has_i7"] = lower_title.str.contains(
        r"\bi7[-\s]?\d*",
        regex=True,
    ).astype(int)

    df["has_i9"] = lower_title.str.contains(
        r"\bi9[-\s]?\d*",
        regex=True,
    ).astype(int)

    df["has_ryzen"] = lower_title.str.contains(
        r"\bryzen\b",
        regex=True,
    ).astype(int)

    # --------------------------------------------------------
    # GPU SIGNALS
    # --------------------------------------------------------

    df["has_rtx"] = lower_title.str.contains(
        r"\brtx\b",
        regex=True,
    ).astype(int)

    df["has_gtx"] = lower_title.str.contains(
        r"\bgtx\b",
        regex=True,
    ).astype(int)

    df["has_geforce"] = lower_title.str.contains(
        r"\bgeforce\b",
        regex=True,
    ).astype(int)

    # --------------------------------------------------------
    # PRODUCT TYPE SIGNALS
    # --------------------------------------------------------

    product_signals = {
        "has_laptop": r"\blaptop\b",
        "has_desktop": r"\bdesktop\b",
        "has_server": r"\bserver\b",
        "has_workstation": r"\bworkstation\b",
        "has_nas": r"\bnas\b",
        "has_ssd": r"\bssd\b",
        "has_monitor": r"\bmonitor\b",
        "has_gaming": r"\bgaming\b",
        "has_professional": r"\bprofessional\b",
        "has_industrial": r"\bindustrial\b",
        "has_heavy_duty": r"\bheavy[\s-]?duty\b",
    }

    for feature, pattern in product_signals.items():

        df[feature] = (
            lower_title
            .str.contains(
                pattern,
                regex=True,
            )
            .astype(int)
        )

    # --------------------------------------------------------
    # MODEL / TECHNICAL COMPLEXITY
    # --------------------------------------------------------

    df["number_count"] = title.apply(
        lambda x:
        len(
            re.findall(
                r"\d+(?:\.\d+)?",
                str(x),
            )
        )
    )

    df["alphanumeric_token_count"] = title.apply(
        lambda x:
        len(
            re.findall(
                r"\b(?=\w*[a-zA-Z])(?=\w*\d)\w+\b",
                str(x),
            )
        )
    )

    return df

In [6]:
ATTRIBUTE_NUMERIC_FEATURES = [

    "gb_value",
    "tb_value",
    "storage_capacity_gb",

    "screen_size_inches",

    "wattage",
    "voltage",
    "va_rating",

    "capacity_liters",
    "pack_count",

    "model_year",
    "bay_count",

    "has_i3",
    "has_i5",
    "has_i7",
    "has_i9",
    "has_ryzen",

    "has_rtx",
    "has_gtx",
    "has_geforce",

    "has_laptop",
    "has_desktop",
    "has_server",
    "has_workstation",
    "has_nas",
    "has_ssd",
    "has_monitor",
    "has_gaming",
    "has_professional",
    "has_industrial",
    "has_heavy_duty",

    "number_count",
    "alphanumeric_token_count",
]

In [7]:
# ============================================================
# DEFINE FEATURE LISTS
# RUN THIS CELL BEFORE YOUR CURRENT CELL [5]
# ============================================================

BASE_NUMERIC_FEATURES = [
    "stars",
    "reviews_log1p",
    "bought_log1p",
    "isBestSeller",

    "title_char_length",
    "title_word_count",
    "title_digit_count",
    "title_uppercase_count",
    "title_first_number",

    "has_gb",
    "has_tb",
    "has_ram",
    "has_inch",
    "has_cm",
    "has_kg",
    "has_gram",
    "has_watt",
    "has_volt",
    "has_pack",
    "has_multipack_number",

    "has_pro",
    "has_max",
    "has_premium",
    "has_professional",
    "has_wireless",
    "has_smart",
]


ATTRIBUTE_NUMERIC_FEATURES = [
    "gb_value",
    "tb_value",
    "storage_capacity_gb",
    "screen_size_inches",

    "wattage",
    "voltage",
    "va_rating",
    "capacity_liters",
    "pack_count",

    "model_year",
    "bay_count",

    "has_i3",
    "has_i5",
    "has_i7",
    "has_i9",
    "has_ryzen",

    "has_rtx",
    "has_gtx",
    "has_geforce",

    "has_laptop",
    "has_desktop",
    "has_server",
    "has_workstation",
    "has_nas",
    "has_ssd",
    "has_monitor",
    "has_gaming",
    "has_professional",
    "has_industrial",
    "has_heavy_duty",

    "number_count",
    "alphanumeric_token_count",
]

print("BASE features:", len(BASE_NUMERIC_FEATURES))
print("ATTRIBUTE features:", len(ATTRIBUTE_NUMERIC_FEATURES))
print("✅ Feature lists defined")

BASE features: 26
ATTRIBUTE features: 32
✅ Feature lists defined


In [8]:
numeric_features = (
    BASE_NUMERIC_FEATURES
    + ATTRIBUTE_NUMERIC_FEATURES
    + ["cluster_id"]
)

categorical_features = [
    "category_name",
    "extracted_brand",
]

print("Numeric features:", len(numeric_features))
print("Categorical features:", categorical_features)

Numeric features: 59
Categorical features: ['category_name', 'extracted_brand']


In [9]:
from pathlib import Path
import pandas as pd

CURRENT_DIRECTORY = Path.cwd()

PROJECT_ROOT = (
    CURRENT_DIRECTORY.parent
    if CURRENT_DIRECTORY.name.lower() == "notebooks"
    else CURRENT_DIRECTORY
)

MODEL_INPUT_ROOT = (
    PROJECT_ROOT
    / "data"
    / "amazon_multimodal"
    / "model_input"
)

train_df = pd.read_parquet(
    MODEL_INPUT_ROOT / "train.parquet"
)

validation_df = pd.read_parquet(
    MODEL_INPUT_ROOT / "validation.parquet"
)

test_df = pd.read_parquet(
    MODEL_INPUT_ROOT / "test.parquet"
)

print("Train:", train_df.shape)
print("Validation:", validation_df.shape)
print("Test:", test_df.shape)

Train: (13984, 26)
Validation: (2997, 26)
Test: (2997, 26)


In [10]:
import re
import numpy as np
import pandas as pd


# ============================================================
# BASE FEATURE FUNCTION
# ============================================================

def create_features(df):
    df = df.copy()

    df["title"] = (
        df["title"]
        .fillna("")
        .astype(str)
    )

    df["category_name"] = (
        df["category_name"]
        .fillna("Unknown")
        .astype(str)
    )

    df["stars"] = pd.to_numeric(
        df["stars"],
        errors="coerce",
    ).fillna(0)

    reviews = pd.to_numeric(
        df["reviews"],
        errors="coerce",
    ).fillna(0)

    bought = pd.to_numeric(
        df["boughtInLastMonth"],
        errors="coerce",
    ).fillna(0)

    df["reviews_log1p"] = np.log1p(reviews)
    df["bought_log1p"] = np.log1p(bought)

    df["isBestSeller"] = (
        df["isBestSeller"]
        .fillna(False)
        .astype(int)
    )

    df["cluster_id"] = pd.to_numeric(
        df["cluster_id"],
        errors="coerce",
    ).fillna(-1)

    # --------------------------------------------------------
    # BASIC TITLE FEATURES
    # --------------------------------------------------------

    df["title_char_length"] = (
        df["title"].str.len()
    )

    df["title_word_count"] = (
        df["title"]
        .str.split()
        .str.len()
        .fillna(0)
    )

    df["title_digit_count"] = (
        df["title"]
        .str.count(r"\d")
    )

    df["title_uppercase_count"] = (
        df["title"].apply(
            lambda text: sum(
                ch.isupper()
                for ch in text
            )
        )
    )

    def first_number(text):
        matches = re.findall(
            r"\d+(?:\.\d+)?",
            str(text),
        )

        if not matches:
            return 0.0

        try:
            return float(matches[0])
        except:
            return 0.0

    df["title_first_number"] = (
        df["title"]
        .apply(first_number)
    )

    # --------------------------------------------------------
    # EXISTING TITLE SIGNALS
    # --------------------------------------------------------

    lower_title = (
        df["title"]
        .str.lower()
    )

    patterns = {
        "has_gb": r"\b\d+(?:\.\d+)?\s*gb\b",
        "has_tb": r"\b\d+(?:\.\d+)?\s*tb\b",
        "has_ram": r"\b(?:ram|memory)\b",
        "has_inch": r'\b\d+(?:\.\d+)?\s*(?:inch|inches|")',
        "has_cm": r"\b\d+(?:\.\d+)?\s*cm\b",
        "has_kg": r"\b\d+(?:\.\d+)?\s*kg\b",
        "has_gram": r"\b\d+(?:\.\d+)?\s*(?:g|gram|grams)\b",
        "has_watt": r"\b\d+(?:\.\d+)?\s*(?:w|watt|watts)\b",
        "has_volt": r"\b\d+(?:\.\d+)?\s*(?:v|volt|volts)\b",
        "has_pack": r"\b(?:pack|set|pair|bundle)\b",
        "has_multipack_number": r"\b\d+\s*[- ]?(?:pack|piece|pcs|count|ct)\b",
        "has_pro": r"\bpro\b",
        "has_max": r"\bmax\b",
        "has_premium": r"\bpremium\b",
        "has_professional": r"\bprofessional\b",
        "has_wireless": r"\bwireless\b",
        "has_smart": r"\bsmart\b",
    }

    for feature_name, pattern in patterns.items():
        df[feature_name] = (
            lower_title
            .str.contains(
                pattern,
                regex=True,
            )
            .astype(int)
        )

    return df


print("✅ create_features() defined")

✅ create_features() defined


In [11]:
train_df = create_features(train_df)
validation_df = create_features(validation_df)
test_df = create_features(test_df)

train_df = enrich_title_features(train_df)
validation_df = enrich_title_features(validation_df)
test_df = enrich_title_features(test_df)

In [12]:
print("Train columns after engineering:", len(train_df.columns))

missing_train = [
    col
    for col in numeric_features + categorical_features
    if col not in train_df.columns
]

if missing_train:
    print("❌ Missing features:")
    for col in missing_train:
        print(" -", col)
else:
    print("✅ ALL FEATURES EXIST")

Train columns after engineering: 82
✅ ALL FEATURES EXIST


In [14]:
# ============================================================
# FINAL FEATURE LISTS - REMOVE DUPLICATES
# ============================================================

numeric_features = list(
    dict.fromkeys(
        BASE_NUMERIC_FEATURES
        + ATTRIBUTE_NUMERIC_FEATURES
        + ["cluster_id"]
    )
)

categorical_features = [
    "category_name",
    "extracted_brand",
]

print("Numeric features:", len(numeric_features))
print("Categorical features:", len(categorical_features))


# ============================================================
# CHECK FOR DUPLICATES
# ============================================================

from collections import Counter

all_raw_numeric = (
    BASE_NUMERIC_FEATURES
    + ATTRIBUTE_NUMERIC_FEATURES
    + ["cluster_id"]
)

duplicates = [
    name
    for name, count in Counter(all_raw_numeric).items()
    if count > 1
]

print("\nDuplicate features found:", duplicates)

assert len(numeric_features) == len(set(numeric_features))

print("✅ Final numeric feature list contains no duplicates")

Numeric features: 58
Categorical features: 2

Duplicate features found: ['has_professional']
✅ Final numeric feature list contains no duplicates


In [15]:
required_features = (
    numeric_features
    + categorical_features
)

missing_features = [
    col
    for col in required_features
    if col not in train_df.columns
]

duplicate_features = [
    col
    for col, count in Counter(required_features).items()
    if count > 1
]

print("=" * 70)
print("FINAL FEATURE VALIDATION")
print("=" * 70)

print("Numeric:", len(numeric_features))
print("Categorical:", len(categorical_features))
print("Total:", len(required_features))
print("Missing:", missing_features)
print("Duplicates:", duplicate_features)

if not missing_features and not duplicate_features:
    print("\n✅ FEATURE CONFIGURATION PASSED")
else:
    print("\n❌ DO NOT TRAIN YET")

FINAL FEATURE VALIDATION
Numeric: 58
Categorical: 2
Total: 60
Missing: []
Duplicates: []

✅ FEATURE CONFIGURATION PASSED


# preprocessing + TF-IDF + Log-LightGBM training

In [16]:
from scipy.sparse import csr_matrix, hstack
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    median_absolute_error,
    r2_score,
)
import lightgbm as lgb
import numpy as np
import pandas as pd
import joblib


# ============================================================
# PREPROCESSOR
# ============================================================

preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            StandardScaler(),
            numeric_features,
        ),
        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False,
            ),
            categorical_features,
        ),
    ]
)

X_train_structured = (
    preprocessor
    .fit_transform(train_df)
    .astype(np.float32)
)

X_validation_structured = (
    preprocessor
    .transform(validation_df)
    .astype(np.float32)
)

X_test_structured = (
    preprocessor
    .transform(test_df)
    .astype(np.float32)
)

print(
    "Structured features:",
    X_train_structured.shape
)


# ============================================================
# TF-IDF
# ============================================================

tfidf = TfidfVectorizer(
    lowercase=True,
    strip_accents="unicode",
    ngram_range=(1, 2),
    min_df=3,
    max_df=0.98,
    max_features=8000,
    sublinear_tf=True,
    dtype=np.float32,
)

X_train_tfidf = tfidf.fit_transform(
    train_df["title"]
)

X_validation_tfidf = tfidf.transform(
    validation_df["title"]
)

X_test_tfidf = tfidf.transform(
    test_df["title"]
)

print(
    "TF-IDF features:",
    X_train_tfidf.shape
)


# ============================================================
# COMBINE
# ============================================================

X_train = hstack(
    [
        csr_matrix(
            X_train_structured
        ),
        X_train_tfidf,
    ],
    format="csr",
)

X_validation = hstack(
    [
        csr_matrix(
            X_validation_structured
        ),
        X_validation_tfidf,
    ],
    format="csr",
)

X_test = hstack(
    [
        csr_matrix(
            X_test_structured
        ),
        X_test_tfidf,
    ],
    format="csr",
)

print(
    "Final train shape:",
    X_train.shape
)


# ============================================================
# TARGET
# ============================================================

y_train = (
    train_df["price"]
    .astype(np.float32)
    .to_numpy()
)

y_validation = (
    validation_df["price"]
    .astype(np.float32)
    .to_numpy()
)

y_test = (
    test_df["price"]
    .astype(np.float32)
    .to_numpy()
)

y_train_log = np.log1p(
    y_train
)

y_validation_log = np.log1p(
    y_validation
)


# ============================================================
# TRAIN LIGHTGBM
# ============================================================

print()
print("=" * 80)
print("TRAINING ATTRIBUTE-ENRICHED LOG-LIGHTGBM V3")
print("=" * 80)

model = lgb.LGBMRegressor(
    objective="regression",

    n_estimators=4000,

    learning_rate=0.025,

    num_leaves=63,

    max_depth=-1,

    min_child_samples=20,

    subsample=0.85,

    colsample_bytree=0.85,

    reg_alpha=0.05,

    reg_lambda=1.0,

    random_state=42,

    verbosity=-1,

    n_jobs=-1,
)

model.fit(
    X_train,
    y_train_log,

    eval_set=[
        (
            X_validation,
            y_validation_log,
        )
    ],

    callbacks=[
        lgb.early_stopping(
            stopping_rounds=150,
            verbose=False,
        )
    ],
)


# ============================================================
# PREDICTIONS
# ============================================================

validation_log_prediction = (
    model.predict(
        X_validation
    )
)

test_log_prediction = (
    model.predict(
        X_test
    )
)

validation_prediction = np.expm1(
    validation_log_prediction
)

test_prediction = np.expm1(
    test_log_prediction
)

validation_prediction = np.clip(
    validation_prediction,
    0,
    None,
)

test_prediction = np.clip(
    test_prediction,
    0,
    None,
)


# ============================================================
# METRICS
# ============================================================

def evaluate(
    name,
    y_true,
    prediction,
):

    mae = mean_absolute_error(
        y_true,
        prediction,
    )

    rmse = np.sqrt(
        mean_squared_error(
            y_true,
            prediction,
        )
    )

    median_ae = (
        median_absolute_error(
            y_true,
            prediction,
        )
    )

    r2 = r2_score(
        y_true,
        prediction,
    )

    print()
    print(name)
    print("-" * 60)

    print(
        f"MAE:       {mae:.4f}"
    )

    print(
        f"RMSE:      {rmse:.4f}"
    )

    print(
        f"Median AE: {median_ae:.4f}"
    )

    print(
        f"R²:        {r2:.4f}"
    )

    return {
        "mae": mae,
        "rmse": rmse,
        "median_ae": median_ae,
        "r2": r2,
    }


validation_metrics = evaluate(
    "Validation",
    y_validation,
    validation_prediction,
)

test_metrics = evaluate(
    "Test",
    y_test,
    test_prediction,
)


# ============================================================
# COMPARE WITH CURRENT BEST
# ============================================================

comparison_df = pd.DataFrame(
    [
        {
            "model":
                "Current Best V2",

            "validation_mae":
                21.8504,

            "validation_rmse":
                85.0096,

            "validation_median_ae":
                8.0066,

            "validation_r2":
                0.4190,
        },

        {
            "model":
                "Attribute-Enriched V3",

            "validation_mae":
                validation_metrics[
                    "mae"
                ],

            "validation_rmse":
                validation_metrics[
                    "rmse"
                ],

            "validation_median_ae":
                validation_metrics[
                    "median_ae"
                ],

            "validation_r2":
                validation_metrics[
                    "r2"
                ],
        },
    ]
)

print()
print("=" * 80)
print("V2 vs V3")
print("=" * 80)

display(
    comparison_df
)


# ============================================================
# SAVE
# ============================================================

MODEL_ROOT = (
    PROJECT_ROOT
    / "models"
    / "price_prediction"
    / "attribute_enriched_v3"
)

REPORT_ROOT = (
    PROJECT_ROOT
    / "data"
    / "reports"
    / "price_prediction"
    / "attribute_enriched_v3"
)

MODEL_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

REPORT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

joblib.dump(
    model,
    MODEL_ROOT
    / "attribute_enriched_log_lightgbm.joblib",
)

joblib.dump(
    preprocessor,
    MODEL_ROOT
    / "structured_preprocessor.joblib",
)

joblib.dump(
    tfidf,
    MODEL_ROOT
    / "title_tfidf.joblib",
)

comparison_df.to_csv(
    REPORT_ROOT
    / "v2_vs_v3.csv",
    index=False,
)

print()
print("✅ V3 training completed")

Structured features: (13984, 337)
TF-IDF features: (13984, 8000)
Final train shape: (13984, 8337)

TRAINING ATTRIBUTE-ENRICHED LOG-LIGHTGBM V3


/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)



Validation
------------------------------------------------------------
MAE:       21.9156
RMSE:      84.0580
Median AE: 7.9655
R²:        0.4319

Test
------------------------------------------------------------
MAE:       21.4789
RMSE:      65.0983
Median AE: 7.8972
R²:        0.5283

V2 vs V3


,model,validation_mae,validation_rmse,validation_median_ae,validation_r2
0,Current Best V2,21.850400,85.00960,8.00660,0.419000
1,Attribute-Enriched V3,21.915557,84.05805,7.96545,0.431929



✅ V3 training completed
